# Edge truncation effects (Delaunay, kNN)

Example: one MIBI-TOF image cropped to its central 3/4; the graph built on the crop gains edges that the
global graph does not have (red). Summary over all datasets from `results/computational_effects/edge_effects/`
(`src/computational_effects/edge_effects.py`). The example crop is recomputed here (one image, seconds).

In [ ]:
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path("../..")
sys.path.insert(0, str((ROOT / "src" / "computational_effects").resolve()))
from border_effects_kNN_del import trunc_graphs

plt.rcParams["svg.fonttype"] = "none"
RES = ROOT / "results" / "computational_effects" / "edge_effects"
all_results_del = pd.read_csv(RES / "del_trunc.csv")
all_results_knn = pd.read_csv(RES / "kNN_trunc.csv")

## Example crop

In [ ]:
with open(ROOT / "mibitof_coords" / "coords.pickle", "rb") as f:
    datasets = pickle.load(f)

EXAMPLE = "glioma_mibitof:Brainiaqc_R1C1_whole_cell.tiff"
SQ = (1 / 8, 7 / 8)  # crop: central 3/4 of the image, as in edge_effects.py
coords = datasets[EXAMPLE]
w, h = np.ptp(coords[:, 0]), np.ptp(coords[:, 1])
xlim = (coords[:, 0].min() + w * SQ[0], coords[:, 0].min() + w * SQ[1])
ylim = (coords[:, 1].min() + h * SQ[0], coords[:, 1].min() + h * SQ[1])
out = trunc_graphs(coords=coords, return_graphs=True, method="delaunay", xlim=xlim, ylim=ylim)
coords_ex = out["coords"]
G, subG, plotG, truncG = out["G"], out["subG"], out["plotG"], out["truncG"]

f, axs = plt.subplots(1, 2, figsize=(6, 3))
nx.draw(G, pos=coords_ex, edge_color=(0.5, 0.5, 0.5, 0.5), node_size=1, node_color="gray", ax=axs[0])
nx.draw(subG, pos=coords_ex, node_size=1, node_color="blue", ax=axs[0])
nx.draw_networkx_edges(plotG, pos=coords_ex, ax=axs[1], edge_color="gray")
nx.draw(subG, pos=coords_ex, ax=axs[1], node_size=1, node_color="blue")
in_global = [e in G.edges for e in truncG.edges]
nx.draw_networkx_edges(truncG, pos=coords_ex, ax=axs[1], edge_color=np.where(in_global, "black", "red"))
for ax in axs:
    ax.set_aspect("equal")
plt.tight_layout()
plt.savefig("edge_effects_example.svg")

In [ ]:
sns.violinplot(out["edge_df"], y="Edge length", hue="New edge", cut=0, fill=False, split=True,
               palette={False: "black", True: "red"}, hue_order=[True, False], inner=None)
plt.savefig("edge_effects_example_lengths.svg")

## All datasets

In [ ]:
f, axs = plt.subplots(1, 5, figsize=(10, 2))
sns.violinplot(all_results_del, y="% new edges", ax=axs[0], legend=False, fill=False, color="purple")
sns.violinplot(all_results_del, y="P_adj", ax=axs[1], legend=False, fill=False, color="purple")
sns.violinplot(all_results_knn, hue="k", y="% new edges", ax=axs[2], legend=False, fill=False, color="green")
sns.violinplot(all_results_knn, hue="k", y="P_MWU_adj", ax=axs[3], legend=False, fill=False, color="green")
sns.violinplot(all_results_knn, hue="k", y="P_fisher_adj", ax=axs[4], legend=True, fill=False, color="green")
axs[0].set_title("$\\mathbf{d}$ % of edges that\nwere not contained\nin global graph")
axs[1].set_title("$\\mathbf{e}$ Adjusted MWU P-values\nof original vs. new\nedge length")
axs[2].set_title("$\\mathbf{f}$ % of edges that\nwere not contained\nin global graph")
axs[3].set_title("$\\mathbf{g}$ Adjusted MWU P-values\nof original vs. new\nedge length")
axs[4].set_title("$\\mathbf{h}$ Adjusted Fisher's exact\nP-values of original vs.\nnew edge symmetry")
plt.tight_layout()
plt.savefig("edge_effects_summary.svg", bbox_inches="tight")
print(f"Delaunay % new edges median: {all_results_del['% new edges'].median():.4f}")
for k, sub in all_results_knn.groupby("k"):
    print(f"kNN k={k}: max={sub['% new edges'].max():.4f}, median={sub['% new edges'].median():.4f}")